# Qwen3-VL Screenshot Icon Finder

Upload an application screenshot, describe the icon or control you want, and receive its bounding box and center point in the **original image's pixel coordinates**.

This is a zero-shot feasibility test using `Qwen/Qwen3-VL-2B-Instruct`; it does not train a model. The model runs locally inside the Colab runtime in memory-safe 4-bit mode. After the workflow is stable, you can optionally try the 4B model on a larger GPU.

Use this order in Google Colab:

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Choose **Runtime → Run all** so imports, the model, and helper functions are initialized in order.
3. Upload a screenshot when prompted, then edit `TARGET` to locate another control.

## Setup

Install the official Qwen3-VL-compatible Transformers version and the packages needed for 4-bit inference.

In [ ]:
!pip -q install -U "transformers>=4.57.0" "accelerate>=1.0.0" bitsandbytes "qwen-vl-utils==0.0.14" pillow matplotlib

### Check the GPU

In [ ]:
import ast
import gc
import json
import os
import re
import time
from pathlib import Path
from urllib.parse import unquote, urlparse

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from IPython.display import JSON, display
from PIL import Image, ImageDraw

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In Colab, select Runtime → Change runtime type → T4 GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)

### Load Qwen3-VL

The 2B model is the safer default for a Colab T4. It is loaded in NF4 4-bit mode, and any previously loaded model is removed before reloading so rerunning this cell does not temporarily keep two models in memory. The first run downloads the model weights.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# Remove an earlier model before re-running this cell. Loading a replacement
# while the old model is still referenced can crash a small Colab runtime.
for variable_name in ("model", "processor", "quantization_config"):
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
).eval()

print("Loaded:", MODEL_ID)
free_bytes, total_bytes = torch.cuda.mem_get_info()
print("Memory allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1e9))
print("GPU memory free: %.2f / %.2f GB" % (free_bytes / 1e9, total_bytes / 1e9))

## Upload a screenshot

Upload a PNG, JPG, JPEG, or WebP screenshot. You can rerun this cell for another application. If this cell is skipped or Colab removes the old file, the detection cell will safely ask you to upload again.

In [ ]:
from pathlib import Path

from google.colab import files
from IPython.display import display
from PIL import Image

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No screenshot was uploaded.")

IMAGE_PATH = str(Path(next(iter(uploaded))).resolve())
image = Image.open(IMAGE_PATH).convert("RGB")
print(f"Uploaded: {IMAGE_PATH} ({image.width} × {image.height} pixels)")
display(image)

## Detection functions

The prompt requests coordinates on a normalized 0–1000 canvas. The code converts them back to the original screenshot's pixels, clamps invalid values, and draws the result.

In [ ]:
# Repeat the lightweight imports here so this helper cell remains safe after
# partial or out-of-order Colab execution. Model loading still happens above.
import ast
import gc
import json
import time
from pathlib import Path
from urllib.parse import unquote, urlparse

from IPython.display import JSON, display
from PIL import Image, ImageDraw

MIN_IMAGE_PIXELS = 256 * 256
MAX_IMAGE_PIXELS = 1024 * 1024
MAX_NEW_TOKENS = 128
USE_GENERATION_CACHE = True
SUPPORTED_IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def clear_runtime_memory():
    gc.collect()
    if "torch" in globals() and torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_memory_status():
    if "torch" not in globals() or not torch.cuda.is_available():
        return "GPU unavailable"
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    return (
        f"GPU free {free_bytes / 1e9:.2f}/{total_bytes / 1e9:.2f} GB; "
        f"allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB"
    )

def _resolve_local_image_path(image_path):
    if image_path is None:
        raise FileNotFoundError("No image path was provided.")

    value = unquote(str(image_path).strip())
    parsed = urlparse(value)
    if parsed.scheme == "file":
        value = unquote(parsed.path)

    candidates = [Path(value)]
    if not Path(value).is_absolute():
        candidates.append(Path("/content") / value)

    for candidate in candidates:
        if candidate.is_file():
            return str(candidate.resolve())

    requested_name = Path(value).name
    requested_stem = Path(requested_name).stem.split(" (")[0]
    possible_matches = []
    content_dir = Path("/content")
    if content_dir.exists():
        for candidate in content_dir.iterdir():
            if candidate.is_file() and candidate.suffix.lower() in SUPPORTED_IMAGE_SUFFIXES:
                if candidate.name == requested_name or candidate.stem.startswith(requested_stem):
                    possible_matches.append(candidate)

    if possible_matches:
        newest = max(possible_matches, key=lambda path: path.stat().st_mtime)
        return str(newest.resolve())

    available = []
    if content_dir.exists():
        available = [
            item.name for item in content_dir.iterdir()
            if item.is_file() and item.suffix.lower() in SUPPORTED_IMAGE_SUFFIXES
        ]
    raise FileNotFoundError(
        f"Screenshot not found: {value}. Available images in /content: {available}"
    )

def upload_screenshot():
    global IMAGE_PATH
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No screenshot was uploaded.")

    IMAGE_PATH = _resolve_local_image_path(next(iter(uploaded)))
    uploaded_image = Image.open(IMAGE_PATH).convert("RGB")
    print(f"Using: {IMAGE_PATH}")
    print(f"Size: {uploaded_image.width} × {uploaded_image.height} pixels")
    display(uploaded_image)
    return IMAGE_PATH

def ensure_image_path():
    global IMAGE_PATH
    current_path = globals().get("IMAGE_PATH")
    if current_path is not None:
        try:
            IMAGE_PATH = _resolve_local_image_path(current_path)
            return IMAGE_PATH
        except FileNotFoundError:
            print("The previous screenshot is no longer available in this runtime.")

    print("Choose a screenshot to continue.")
    return upload_screenshot()

def check_runtime_ready():
    required = [
        "model", "processor", "process_vision_info", "MODEL_ID", "torch",
        "Image", "ImageDraw", "Path", "unquote", "urlparse",
        "JSON", "display", "time", "json", "ast", "gc",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            "Runtime state is incomplete. Restart the runtime and use Runtime → Run all. "
            f"Missing: {missing}"
        )
    if not torch.cuda.is_available():
        raise RuntimeError("A GPU runtime is required. Select Runtime → Change runtime type → T4 GPU.")

def _extract_json_object(text):
    cleaned = text.strip().replace("```json", "").replace("```", "")
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start < 0 or end <= start:
        raise ValueError(f"The model did not return a JSON object. Raw response: {text}")
    candidate = cleaned[start:end + 1]
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        try:
            value = ast.literal_eval(candidate)
            if isinstance(value, dict):
                return value
        except (ValueError, SyntaxError):
            pass
        raise ValueError(f"Could not parse the model response as JSON. Raw response: {text}")

def _coordinate_mode(values, declared_space, image_width, image_height):
    space = str(declared_space or "").lower()
    maximum = max(abs(float(value)) for value in values)
    if "normalized_1000" in space or "0-1000" in space:
        return "normalized_1000"
    if "pixel" in space or maximum > 1000:
        return "pixels"
    if "0-1" in space or "normalized_1" in space or maximum <= 1.5:
        return "normalized_1"
    return "normalized_1000"

def _point_to_pixels(point, mode, width, height):
    x, y = map(float, point[:2])
    if mode == "normalized_1":
        x, y = x * width, y * height
    elif mode == "normalized_1000":
        x, y = x * width / 1000.0, y * height / 1000.0
    return [round(min(max(x, 0), width - 1)), round(min(max(y, 0), height - 1))]

def _box_to_pixels(box, mode, width, height):
    x1, y1, x2, y2 = map(float, box[:4])
    if mode == "normalized_1":
        x1, x2 = x1 * width, x2 * width
        y1, y2 = y1 * height, y2 * height
    elif mode == "normalized_1000":
        x1, x2 = x1 * width / 1000.0, x2 * width / 1000.0
        y1, y2 = y1 * height / 1000.0, y2 * height / 1000.0
    x1, x2 = sorted((min(max(x1, 0), width - 1), min(max(x2, 0), width - 1)))
    y1, y2 = sorted((min(max(y1, 0), height - 1), min(max(y2, 0), height - 1)))
    return [round(x1), round(y1), round(x2), round(y2)]

def _normalize_detection(payload, width, height):
    status = str(payload.get("status", "found")).lower()
    raw_box = payload.get("bbox_2d") or payload.get("bbox") or payload.get("box")
    raw_point = payload.get("point_2d") or payload.get("point") or payload.get("center")
    declared_space = payload.get("coordinate_space", "normalized_1000")

    pixel_box = None
    pixel_center = None
    if isinstance(raw_box, (list, tuple)) and len(raw_box) >= 4:
        mode = _coordinate_mode(raw_box, declared_space, width, height)
        pixel_box = _box_to_pixels(raw_box, mode, width, height)
        x1, y1, x2, y2 = pixel_box
        pixel_center = [round((x1 + x2) / 2), round((y1 + y2) / 2)]
    elif isinstance(raw_point, (list, tuple)) and len(raw_point) >= 2:
        mode = _coordinate_mode(raw_point, declared_space, width, height)
        pixel_center = _point_to_pixels(raw_point, mode, width, height)

    if status == "found" and pixel_center is None:
        status = "parse_failed"

    confidence = payload.get("confidence")
    try:
        confidence = float(confidence) if confidence is not None else None
    except (TypeError, ValueError):
        confidence = None

    return {
        "status": status,
        "label": payload.get("label"),
        "box": pixel_box,
        "center": pixel_center,
        "self_reported_confidence": confidence,
        "reason": payload.get("reason"),
        "image_size": [width, height],
    }

def find_icon(image_path, target):
    check_runtime_ready()
    clear_runtime_memory()
    print("Before inference:", gpu_memory_status())
    image_path = _resolve_local_image_path(image_path)
    source_image = Image.open(image_path).convert("RGB")
    width, height = source_image.size
    prompt = f"""
You are a precise GUI grounding system. Locate this requested interface element:
TARGET: {target}

Use the entire screenshot. Distinguish the active application's real controls from icons inside photos, videos, advertisements, thumbnails, or decorative content. If the request says main or primary, prefer the control belonging to the active/main content area.

Return exactly one JSON object and no markdown. Status must be found, not_found, or ambiguous. For not_found, use null for bbox_2d and point_2d:
{{
  "status": "found",
  "label": "short description",
  "bbox_2d": [x1, y1, x2, y2],
  "point_2d": [center_x, center_y],
  "coordinate_space": "normalized_1000",
  "confidence": number from 0 to 1,
  "reason": "brief visual reason"
}}

All coordinates must be relative to the FIRST full screenshot on a 0-to-1000 scale: top-left is [0,0], bottom-right is [1000,1000]. Make the box tightly cover the visible clickable control, including its button background when present. The original screenshot is {width} by {height} pixels.
""".strip()

    messages = [{
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": source_image,
                "min_pixels": MIN_IMAGE_PIXELS,
                "max_pixels": MAX_IMAGE_PIXELS,
            },
            {"type": "text", "text": prompt},
        ],
    }]

    inputs = None
    image_inputs = None
    video_inputs = None
    generated_ids = None
    peak_memory_gb = None
    try:
        chat_text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        patch_size = getattr(processor.image_processor, "patch_size", 16)
        image_inputs, video_inputs = process_vision_info(
            messages, image_patch_size=patch_size
        )
        inputs = processor(
            text=[chat_text],
            images=image_inputs,
            videos=video_inputs,
            do_resize=False,
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        started = time.perf_counter()
        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=USE_GENERATION_CACHE,
            )
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        peak_memory_gb = round(torch.cuda.max_memory_allocated() / 1e9, 3)

        trimmed_ids = [
            output_ids[len(input_ids):].detach().cpu()
            for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
        ]
        raw_response = processor.batch_decode(
            trimmed_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
    except torch.OutOfMemoryError as error:
        raise RuntimeError(
            "GPU memory was exhausted. Restart the Colab runtime, select a T4 GPU, "
            "and use Runtime → Run all. If needed, change MAX_IMAGE_PIXELS to 768 * 768."
        ) from error
    finally:
        del inputs, image_inputs, video_inputs, generated_ids
        clear_runtime_memory()

    try:
        payload = _extract_json_object(raw_response)
        result = _normalize_detection(payload, width, height)
    except ValueError as error:
        result = {
            "status": "parse_failed",
            "label": None,
            "box": None,
            "center": None,
            "self_reported_confidence": None,
            "reason": str(error),
            "image_size": [width, height],
        }

    result["target"] = target
    result["inference_seconds"] = round(elapsed, 3)
    result["peak_gpu_memory_gb"] = peak_memory_gb
    result["model"] = MODEL_ID
    result["raw_response"] = raw_response
    print("After inference:", gpu_memory_status())
    return result

def draw_result(image_path, result, save_path="qwen3_vl_detection.png"):
    image_path = _resolve_local_image_path(image_path)
    annotated = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(annotated)
    box = result.get("box")
    center = result.get("center")
    label = result.get("label") or result.get("target") or "target"

    if box:
        x1, y1, x2, y2 = box
        line_width = max(3, round(min(annotated.size) / 300))
        draw.rectangle(box, outline=(255, 40, 40), width=line_width)
        text_y = max(0, y1 - 18)
        draw.rectangle([x1, text_y, x1 + max(80, len(label) * 7), text_y + 18], fill=(255, 40, 40))
        draw.text((x1 + 3, text_y + 2), label, fill=(255, 255, 255))
    if center:
        cx, cy = center
        radius = max(5, round(min(annotated.size) / 150))
        draw.ellipse([cx - radius, cy - radius, cx + radius, cy + radius], fill=(0, 230, 120), outline=(0, 0, 0))

    annotated.save(save_path)
    display(annotated)
    return annotated

## Find one icon

Change `TARGET` to any visible control. Be specific when the screen contains similar icons—for example, say `main play button in the active video player; ignore recommendation thumbnails`.

In [ ]:
if "find_icon" not in globals() or "ensure_image_path" not in globals():
    raise RuntimeError("Run the Detection functions cell first, or use Runtime → Run all.")

check_runtime_ready()
IMAGE_PATH = ensure_image_path()
TARGET = "main play button in the active video player; ignore recommendation thumbnails"

result = find_icon(IMAGE_PATH, TARGET)
display(JSON({key: value for key, value in result.items() if key != "raw_response"}))
print("Raw model response:")
print(result["raw_response"])
draw_result(IMAGE_PATH, result)

## Try other controls

Run the next cell with one target at a time. Each target requires a separate model inference.

In [ ]:
# Examples:
# TARGET = "pause button belonging to the active media player"
# TARGET = "fullscreen button in the active media player"
# TARGET = "share icon for the currently open post"
# TARGET = "settings gear for the video player"
# TARGET = "close button for the topmost dialog"

if "find_icon" not in globals() or "ensure_image_path" not in globals():
    raise RuntimeError("Run the Detection functions cell first, or use Runtime → Run all.")

check_runtime_ready()
IMAGE_PATH = ensure_image_path()
TARGET = "fullscreen button in the active media player"
result = find_icon(IMAGE_PATH, TARGET)
display(JSON({key: value for key, value in result.items() if key != "raw_response"}))
draw_result(IMAGE_PATH, result, save_path="qwen3_vl_detection_second_target.png")

## Save the result

The confidence value is the model's own estimate and is not statistically calibrated. Judge the test using the marked image and coordinate accuracy.

In [ ]:
if "result" not in globals():
    raise RuntimeError("No detection result exists. Run a detection cell first.")

RESULT_JSON_PATH = "qwen3_vl_detection.json"
with open(RESULT_JSON_PATH, "w", encoding="utf-8") as output_file:
    json.dump(result, output_file, indent=2, ensure_ascii=False)

print("Saved:", RESULT_JSON_PATH)
# Uncomment to download the JSON file:
# files.download(RESULT_JSON_PATH)

## Checks and interpretation

Test at least 50–100 screenshots from different apps before drawing a conclusion. Record whether the center point falls inside the correct control, whether a similar icon was selected instead, and the inference time. The notebook defaults to the 2B model, a `1024 * 1024` image-pixel budget, and a 128-token response for T4 stability. If the runtime still crashes, restart it and reduce `MAX_IMAGE_PIXELS` to `768 * 768`. Do not repeatedly reload the model without running the updated cleanup cell.

If Qwen3-VL understands unusual icons but its boxes are imprecise or inference is too slow, use it as a dataset-labeling or fallback model and train a small detector for the production path.